<img src="아.png" width="500">

In [2]:
'''

class solarCar:
    def __init__(self, HV=100, LV=12, errorCode=None, error=False, **kwargs):
        self.HV = 100
        self.LV = 12
        self.errorCode = errorCode
        self.error = error
        for key, value in kwargs.items():
            setattr(self, key, value)
'''

'\n\nclass solarCar:\n    def __init__(self, HV=100, LV=12, errorCode=None, error=False, **kwargs):\n        self.HV = 100\n        self.LV = 12\n        self.errorCode = errorCode\n        self.error = error\n        for key, value in kwargs.items():\n            setattr(self, key, value)\n'

목표 속도 입력 </br>
↓</br>
현재 속도와 비교</br>
↓</br>
필요 가속도 계산</br>
↓</br>
공기저항/구름저항/경사저항 계산</br>
↓</br>
필요 구동력 계산</br>
↓</br>
필요 바퀴 토크 계산</br>
↓</br>
필요 모터 토크 계산</br>
↓</br>
필요 q축 전류 계산</br>
↓</br>
인버터가 PWM으로 3상 전압 생성</br>
↓</br>
모터 전류 흐름</br>
↓</br>
토크 발생</br>
↓</br>
차량 가속 또는 속도 유지</br>
↓</br>
기계출력 발생</br>
↓</br>
손실 포함해서 배터리 출력 결정</br>
↓</br>
배터리 단자전압/전류 계산</br>
↓</br>
SOC 감소</br>

In [6]:
import math

class Battery:
    def __init__(self, car, soc=1.0, efficiency=0.95):
        self.car = car
        self.soc = soc
        self.efficiency = efficiency
        self.voltage = (1.7*self.soc + 2.5)*32
        self.car.HV = self.voltage
        self.impedance = (math.log(self.soc, 1.14) + 22.5)/1000
        
    def set_HV(self, state):
        if state == 'charge':
            self.car.HV = (1.7*self.soc + 2.5)*32 + self.current * self.impedance
        else:
            self.car.HV = (1.7*self.soc + 2.5)*32 - self.current * self.impedance   
    
    def charge(self, power):
        if self.soc >= 1:
            self.car.cutoff = True
            return
        else:
            self.car.cutoff = False
        
        self.voltage = (1.7*self.soc + 2.5)*32
        self.impedance = (math.log(self.soc, 1.14) + 22.5) / 1000 
        self.current = (-self.voltage+math.sqrt(self.voltage**2+4*self.impedance*power))/(2*self.impedance)
        self.set_HV('charge')
        self.soc += self.efficiency * power * self.car.dt / 10600000
        self.soc = min(self.soc, 1.0)
        
    def discharge(self, power):
        if self.soc <= 0.01:
            self.car.cutoff = True
            return
        else:
            self.car.cutoff = False
        
        self.voltage = (1.7*self.soc + 2.5)*32
        if self.voltage**2 - 4*self.impedance*power < 0:
            self.car.cutoff = True
            return
        else:
            self.car.cutoff = False
            
        self.current = (self.voltage-math.sqrt(self.voltage**2-4*self.impedance*power))/(2*self.impedance)
        self.set_HV('discharge')
        self.soc -= power * self.car.dt / (self.efficiency * 10600000)
        self.soc = max(self.soc, 0.01)

In [9]:
class MPPT:
    def __init__(self, car, n):
        self.car = car
        self.cell_area = n*0.016384
        self.mppt_efficiency = 0.992
        self.cell_efficiency = 0.248
    
    def power_out(self, ghi):
        return self.mppt_efficiency * self.cell_efficiency * self.cell_area * ghi
    
    
class Motor:
    def __init__(self, car):
        self.emf_constant = 1
        self.motor_constant = 1
        self.motor_efficiency = 0.9
        self.wheel_r = 1
        self.car = car
        
    def resistance(self, ele, w_head, w_cross, speed):
        F_roll = self.car.crr * self.car.mass * 9.8 * math.cos(ele)
        F_slope = self.car.mass * 9.8 * math.sin(ele)
        F_aero = 0.5 * self.car.aero * ((speed + w_head)**2 + w_cross**2)

        return F_roll + F_slope + F_aero
    
    def bat2motor_power(self, speed):
        v_rel = speed - self.car.speed
        a = min(max(v_rel / 5, 0), 0.3)
        F_drive = self.car.mass*a + self.resistance(ele=0, w_cross=0, w_head=0, speed=self.car.speed)
        v_for_power = max(self.car.speed, 1.0)
        return F_drive*self.car.speed/self.motor_efficiency

In [10]:
class SolarCar:
    def __init__(self):
        self.HV = 140
        self.mass = 300
        self.crr = 1
        self.aero = 1
        self.cutoff = False
        self.precharge = False
        self.LV = 12
        self.speed = 0
        self.dt = 1 # 프레임 속도(시뮬레이션 속도 단위? 는 1초)
        self.battery = Battery(self)
        self.mppt1 = MPPT(self, 128) # 셀 128개
        self.mppt2 = MPPT(self, 128) # 셀 128개
        self.mppt3 = MPPT(self, 129) # 셀 129개
        self.motor = Motor(self)
        
    def solar_power(self, ghi):
        return (
            self.mppt1.power_out(ghi)
            + self.mppt2.power_out(ghi)
            + self.mppt3.power_out(ghi)
        )

    def step(self, target_speed, ghi):
        # 1. 태양광 발전량
        p_solar = self.solar_power(ghi)

        # 2. 목표 속도까지 가기 위한 모터 요구 전력
        p_motor = self.motor.bat2motor_power(target_speed)

        # 3. 배터리 입출력 계산
        p_net = p_solar - p_motor

        if p_net >= 0:
            self.battery.charge(p_net)
        else:
            self.battery.discharge(-p_net)

        # 4. 데모용 속도 업데이트
        # 실제 물리모델은 아니고 "목표속도로 천천히 따라감" 정도
        v_rel = target_speed - self.speed
        a = min(max(v_rel / 5, 0), 0.3)
        self.speed += a * self.dt

        return {
            "speed": self.speed,
            "soc": self.battery.soc,
            "HV": self.HV,
            "p_solar": p_solar,
            "p_motor": p_motor,
            "p_net": p_net,
            "cutoff": self.cutoff
        }

In [11]:
car = SolarCar()

for t in range(100):
    result = car.step(
        target_speed=60/3.6,  # 60 km/h
        ghi=800               # 대충 맑은 날
    )

    print(
        t,
        "speed:", round(result["speed"] * 3.6, 2), "km/h",
        "SOC:", round(result["soc"], 4),
        "HV:", round(result["HV"], 2),
        "solar:", round(result["p_solar"], 1),
        "motor:", round(result["p_motor"], 1),
        "net:", round(result["p_net"], 1),
        "cutoff:", result["cutoff"]
    )

0 speed: 1.08 km/h SOC: 1.0 HV: 134.4 solar: 1241.5 motor: 0.0 net: 1241.5 cutoff: True
1 speed: 2.16 km/h SOC: 1.0 HV: 134.4 solar: 1241.5 motor: 1010.0 net: 231.4 cutoff: True
2 speed: 3.24 km/h SOC: 0.9999 HV: 134.27 solar: 1241.5 motor: 2020.1 net: -778.7 cutoff: False
3 speed: 4.32 km/h SOC: 0.9997 HV: 134.1 solar: 1241.5 motor: 3030.4 net: -1788.9 cutoff: False
4 speed: 5.4 km/h SOC: 0.9995 HV: 133.92 solar: 1241.5 motor: 4041.0 net: -2799.5 cutoff: False
5 speed: 6.48 km/h SOC: 0.9991 HV: 133.73 solar: 1241.5 motor: 5051.9 net: -3810.4 cutoff: False
6 speed: 7.56 km/h SOC: 0.9986 HV: 133.54 solar: 1241.5 motor: 6063.2 net: -4821.8 cutoff: False
7 speed: 8.64 km/h SOC: 0.998 HV: 133.34 solar: 1241.5 motor: 7075.1 net: -5833.7 cutoff: False
8 speed: 9.72 km/h SOC: 0.9974 HV: 133.14 solar: 1241.5 motor: 8087.7 net: -6846.2 cutoff: False
9 speed: 10.8 km/h SOC: 0.9966 HV: 132.93 solar: 1241.5 motor: 9100.9 net: -7859.5 cutoff: False
10 speed: 11.88 km/h SOC: 0.9957 HV: 132.71 solar: